# ES4304 — Data Access Accounts

To take part in the course you need accounts with three data providers, to download the satellite data the tutorials use.

**You MUST do this before the course starts.** You cannot process any data without these accounts, and sorting them out during a session wastes your time and everyone else's.

| Provider | Needed for | Register at | Approval |
|---|---|---|---|
| **NASA Earthdata** | PACE (2.1), SWOT (2.3), OSCAR (2.4) | <https://urs.earthdata.nasa.gov/users/new> | Immediate |
| **JAXA P-Tree** | Himawari SST (2.2) | <https://www.eorc.jaxa.jp/ptree/registration_top.html> | **Up to several working days** |
| **Copernicus Data Space** | Sentinel-2 (1.1, 1.2), Sentinel-1 (3.1) | <https://dataspace.copernicus.eu/> | Immediate |

All three are free and quick to fill in. The JAXA one cannot be rushed, though — a person approves it, so register now rather than the night before Tutorial 2.

The more fiddly part is telling the notebooks about your accounts. On Colab that is done with **Colab Secrets** — section 2 — and you never type a password into a cell.

## Setting up

Run the cell below first, before anything else on this page.

Colab hands you an empty machine, so it installs the packages this
tutorial needs, and sets up `login()` — which the download cells use to
get your data provider username and password from your Colab Secrets. It takes a minute or two.

Colab does not keep anything between sessions, so run it again whenever you
reopen this notebook — it is safe to re-run at any point.


In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# Colab gives you an empty machine and takes it back when the session ends,
# so this installs the packages the tutorial needs, and sets up how the
# download cells get your data provider logins. Run it again whenever you
# reopen the notebook.

import importlib
import importlib.util
import os
import subprocess
import sys

try:
    import google.colab               # noqa: F401 - importing it is the test
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED = {                      # import name -> pip name
    "earthaccess": "earthaccess>=0.16",
    "requests": "requests",
}

missing = [spec for mod, spec in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=False)
    importlib.invalidate_caches()

    # Check they import, rather than that pip said it was fine: a package can
    # install cleanly and still fail to load, and the fix for that is a
    # restart, which pip cannot do for you.
    broken = []
    for mod in REQUIRED:
        try:
            importlib.import_module(mod)
        except Exception as error:        # noqa: BLE001 - report anything
            broken.append(f"{mod} ({type(error).__name__}: {error})")

    if broken:
        print("\nInstalled, but these still will not import:")
        for line in broken:
            print("   ", line)
        print("Runtime -> Restart session, then run this cell again.")
    else:
        print("Ready.")
else:
    print("Ready - nothing to install.")


# --- Signing in ------------------------------------------------------------
# This defines `login()`. The download cells below call it when they need your
# username and password for a data provider; nothing is asked for here.
#
# It looks in **Colab Secrets** first - the key icon in the left sidebar. A
# secret belongs to your Google account rather than to this notebook, so you
# add it once and every tutorial in the course finds it. Turn on "Notebook
# access" for each. This tutorial asks for:
#
#     EARTHDATA_USERNAME  and  EARTHDATA_PASSWORD
#         your NASA Earthdata login
#     PTREE_USERNAME  and  PTREE_PASSWORD
#         your P-Tree FTP credentials - NOT your P-Tree website login
#     CDSE_USERNAME  and  CDSE_PASSWORD
#         the email address you registered with, and its password
#
# There is no typing them in here instead. A notebook can read Secrets but
# cannot write them, so a password typed into a cell would be gone again next
# session - and saved into the notebook if you shared it. `login()` says what
# to add and where if a secret is missing.

CREDENTIALS = {            # what to call it -> secret names, and sign-up
    "NASA Earthdata":
        ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD",
         "https://urs.earthdata.nasa.gov/users/new"),
    "JAXA P-Tree":
        ("PTREE_USERNAME", "PTREE_PASSWORD",
         "https://www.eorc.jaxa.jp/ptree/registration_top.html"),
    "Copernicus Data Space":
        ("CDSE_USERNAME", "CDSE_PASSWORD",
         "https://dataspace.copernicus.eu/"),
}


def read_secret(name):
    """One Colab secret, or None if unset or not shared with this notebook."""
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata

        return (userdata.get(name) or "").strip() or None
    except Exception:                 # noqa: BLE001 - unset, or not shared
        return None


def login(provider):
    """The username and password for one data provider, from Colab Secrets.

    Nothing is typed into this notebook. Secrets are re-read on every call, so
    correcting one in the sidebar and running the cell again takes effect.
    """
    user_secret, pass_secret, register = CREDENTIALS[provider]
    user, password = read_secret(user_secret), read_secret(pass_secret)

    if not (user and password):
        raise RuntimeError(
            f"{provider}: {user_secret} and {pass_secret} are not readable.\n\n"
            "Open the key icon in the left sidebar - the Secrets tab - and for\n"
            "each of the two:\n"
            "    1. click + Add new secret\n"
            f"    2. put {user_secret} (then {pass_secret}) in Name\n"
            "    3. paste the value in Value\n"
            "    4. switch Notebook access on\n\n"
            "Then run this cell again. If they are already there, it is the\n"
            "Notebook access switch that is off for this notebook.\n\n"
            f"No account yet? Register at {register}")

    return user, password


## 1. Setup

The same cell appears at the top of every notebook in this course. It installs only what is missing, so it costs nothing when there is nothing to do.

In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# The Colab setup cell at the top has already installed these, so this cell
# finds nothing to do. It is left exactly as it is in the Codespaces
# notebook rather than deleted, so the two versions stay comparable.
import importlib.util
import os
import subprocess
import sys

REQUIRED = {                      # import name -> pip name
    "earthaccess": "earthaccess>=0.16",
    "requests": "requests>=2.31",
}

missing = [pip for mod, pip in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=True)
else:
    print("Environment ready - nothing to install.")

## 2. Colab Secrets

In a Codespace you type your passwords into this notebook once, and they are saved to a file in your home directory that every later notebook reads. Colab has no home directory that survives the session, so that approach has nowhere to put anything.

**Colab Secrets** is the replacement, and it is better: a per-account store that belongs to your Google account rather than to any notebook. Add a secret once and every tutorial in the course finds it. It is never part of the notebook file, so there is nothing to scrub and nothing to leak when you share your work.

**You add them by hand, in the Secrets tab.** A notebook can read secrets but cannot create them — that is deliberate, so a notebook someone sends you cannot write into your credential store. Nothing in this course will ask you to type a password into a cell.

### How to add one

1. Click the **key icon** in the left sidebar. The **Secrets** panel opens.
2. Click **+ Add new secret**.
3. Put the name in the **Name** column — exactly as written in the table below, no spaces.
4. Paste the value in the **Value** column.
5. Switch **Notebook access** on for that row. Without it this notebook cannot read the secret, and the error you get says so.

Repeat for each. You are adding six:

| Secret name | What to put in it |
|---|---|
| `EARTHDATA_USERNAME` | Your NASA Earthdata username |
| `EARTHDATA_PASSWORD` | Your NASA Earthdata password |
| `PTREE_USERNAME` | Your JAXA P-Tree **FTP** username — usually your email with `@` replaced by `_` |
| `PTREE_PASSWORD` | Your JAXA P-Tree **FTP** password |
| `CDSE_USERNAME` | The email address you registered with at Copernicus Data Space |
| `CDSE_PASSWORD` | Your Copernicus Data Space password |

> **P-Tree's FTP credentials are not your P-Tree website login.** They arrive in the approval email. This is the single most common thing to get wrong.

**Notebook access is per notebook.** The switch you turn on here covers this notebook only. The first time you open 2.1 or 2.2, turn it on again there for the secrets that tutorial needs — the secret itself is already saved, so it is one switch, not a re-entry.

Then run the cell below. It reports which of the six this notebook can currently read.

> **What Colab Secrets is not.** It is your account's store, on Google's servers, and it is exactly as safe as your Google account. Use it for these course logins. Do not put anything in it you would mind Google holding.


In [ ]:
# Which of the six this notebook can read right now. Values are shown only as
# a length, so this output is safe to leave in the notebook when you share it.
from google.colab import userdata

WANTED = [
    ("NASA Earthdata", "EARTHDATA_USERNAME", "EARTHDATA_PASSWORD"),
    ("JAXA P-Tree", "PTREE_USERNAME", "PTREE_PASSWORD"),
    ("Copernicus Data Space", "CDSE_USERNAME", "CDSE_PASSWORD"),
]


def peek(name):
    """The secret's length, or why it cannot be read."""
    try:
        value = (userdata.get(name) or "").strip()
    except Exception as why:      # noqa: BLE001 - not set, or access is off
        return f"unreadable ({type(why).__name__})"
    return f"{len(value)} characters" if value else "empty"


for label, user_secret, pass_secret in WANTED:
    print(f"{label}")
    for name in (user_secret, pass_secret):
        print(f"    {name:20s} {peek(name)}")

print("\nAnything 'unreadable' is either not added yet, or added with its")
print("Notebook access switch off. Both are fixed in the key icon panel.")


## 3. Check it worked

The three cells below use the credentials for real: they log in, and then move a few bytes of actual data. A wrong password fails here, in a cell that says so, rather than in the middle of a tutorial.


### NASA Earthdata

`earthaccess.login(strategy="environment")` reads the two environment variables the setup cell filled in, and logs in against Earthdata. **This route checks your credentials immediately** — a wrong password raises here, rather than failing later in the middle of a tutorial.

Then we download one real file, about 26 MB, and delete it again. Logging in proves the password is right; downloading proves the account can actually get data.

In [ ]:
import shutil

import os

import earthaccess

# earthaccess.login() cannot be handed a username and password, so the two
# it reads from the environment are set here from your Colab Secrets.
os.environ["EARTHDATA_USERNAME"], os.environ["EARTHDATA_PASSWORD"] = \
    login("NASA Earthdata")

auth = earthaccess.login(strategy="environment")

# `authenticated` is the flag that actually means "logged in". `username` is
# only filled in when the login used a username and password - a token login
# leaves it as None, so printing it alone is misleading.
print("Authenticated:", auth.authenticated)
print("Logged in to Earthdata as:", auth.username or "(token login - no username)")

if auth.username is None:
    print("\nNo username means earthaccess did not use the environment "
          "variables - it")
    print("was already authenticated with an EARTHDATA_TOKEN. Downloads will")
    print("work, but the credentials you gave above have NOT been checked.")
    print("Print it with os.environ.get('EARTHDATA_TOKEN'); if that shows")
    print("anything, remove EARTHDATA_TOKEN from your Colab Secrets (the")
    print("key icon in the left sidebar), then restart the runtime and")
    print("re-run this notebook to test the credentials you gave.")

results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",     # chlorophyll lives in the biogeochemistry suite
    version="3.2",
    granule_name="*MO*0p1*",           # monthly, 0.1 degree
    count=1,
)

if not results:
    print("\nSearch found nothing. That is not an account problem - a retired")
    print("short_name returns an empty list rather than an error. See S01.")
else:
    print("Found:", results[0].data_links()[0].split("/")[-1])

    TEST_DIR = os.path.join(os.getcwd(), "login_check")
    try:
        files = earthaccess.download(results, TEST_DIR)
        print("Downloaded:", os.path.basename(files[0]))
        print("\nNASA Earthdata access works.")
    finally:
        shutil.rmtree(TEST_DIR, ignore_errors=True)

### JAXA P-Tree

`ftplib` has no idea about environment variables, so the cell reads the two the setup cell filled in and hands them over. Every notebook that needs P-Tree does exactly this, which is why you never type the password again.

Skip this cell if your approval has not arrived.

In [ ]:
from ftplib import FTP

FTP_SERVER = "ftp.ptree.jaxa.jp"

ftp_user, ftp_password = login("JAXA P-Tree")
print(f"user {ftp_user!r}, password {len(ftp_password)} chars")

with FTP(FTP_SERVER) as ftp:
    ftp.login(ftp_user, ftp_password)
    print("Connected. Top-level directories:", ftp.nlst()[:5])
    print("\nJAXA P-Tree access works.")

### Copernicus Data Space Ecosystem

CDSE does not take your password on every request. You exchange it for a short-lived **access token**, and downloads carry the token instead — which is why the notebooks that use Sentinel data all start with the same few lines below, reading your credentials out of the environment exactly as the P-Tree cell does.

The exchange is where a wrong password shows up, as `401 Unauthorized`. Searching the catalogue needs no account at all, so — as with Earthdata — we then read the first bytes of a real product, which does.

In [ ]:
import requests

CDSE_HOST = "identity.dataspace.copernicus.eu"
TOKEN_URL = f"https://{CDSE_HOST}/auth/realms/CDSE/protocol/openid-connect/token"

cdse_user, cdse_password = login("Copernicus Data Space")

response = requests.post(TOKEN_URL, timeout=60, data={
    "client_id": "cdse-public",       # the public client every CDSE script uses
    "grant_type": "password",
    "username": cdse_user,
    "password": cdse_password,
    # With two-factor authentication on, uncomment this and put in the current
    # six-digit code from your authenticator app before running the cell.
    # "totp": "123456",
})
response.raise_for_status()           # 401 here means the password is wrong
token = response.json()["access_token"]

print("Logged in to Copernicus Data Space as:", cdse_user)

# The catalogue is open to anyone; downloading is the part that needs the
# account. Take the newest Sentinel-2 scene there is ...
CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

search = requests.get(CATALOGUE, timeout=60, params={
    "$filter": "Collection/Name eq 'SENTINEL-2' and contains(Name,'MSIL2A')",
    "$orderby": "ContentDate/Start desc",
    "$top": 1,
})

# The catalogue goes down from time to time, and when it does this cell is the
# first place you notice - which is misleading, because searching it needs no
# account. Say so, rather than failing on a missing key and leaving you to
# wonder whether your password is wrong.
try:
    found = search.json().get("value")
except ValueError:                        # not JSON at all - an error page
    found = None

if not found:
    raise RuntimeError(
        f"The Copernicus catalogue returned no result (HTTP "
        f"{search.status_code}).\n{search.text[:200]}\n\n"
        "This is not your account - the catalogue is open to anyone, and your "
        "login\nalready worked in the cell above. It is the service being "
        "temporarily down.\nWait a few minutes and run this cell again; check "
        "https://dataspace.copernicus.eu\nif it keeps happening.")

product = found[0]
print("Found:", product["Name"])

# ... and read the first kilobyte of it. A Sentinel-2 scene is about a
# gigabyte; `stream=True` means nothing is fetched until we ask, and leaving
# the `with` block closes the connection, so the rest never arrives.
#
# Note the hostname. `download` is the download service - `zipper` is an older
# name for the same thing, and you will see it in some notebooks. Do not ask
# the `catalogue` host for the file: it redirects here, and `requests` drops
# the Authorization header when a redirect crosses hostnames, so the token
# would be thrown away and you would get a 401 that is nothing to do with you.
DOWNLOAD = "https://download.dataspace.copernicus.eu/odata/v1/Products({0})/$value"

with requests.get(DOWNLOAD.format(product["Id"]), timeout=60, stream=True,
                  headers={"Authorization": f"Bearer {token}"}) as file_stream:
    file_stream.raise_for_status()
    head = next(file_stream.iter_content(1024))

print(f"Read the first {len(head)} bytes of it.")
print("\nCopernicus Data Space access works.")

## If something failed

- **`KeyError: 'PTREE_USERNAME'`, or similar.** The setup cell at the top has not run in this session — the runtime was restarted, or reconnected after being idle. Run it again; it is safe to re-run.
- **The setup cell asked you rather than reading a secret.** The secret is missing, misspelled, or its **Notebook access** toggle is off for this notebook. Open the key icon in the left sidebar and check all six.
- **A password you typed at the prompt was wrong.** Correct it in Colab Secrets and re-run the setup cell — secrets are re-read every time. If you would rather type it again, **Runtime → Restart session** first, which clears what you typed.
- **`LoginAttemptFailure`.** Earthdata rejected the username or password. Check them by logging in at <https://urs.earthdata.nasa.gov>, then correct `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` in your Colab Secrets and run the setup cell again. It signs you in to each provider separately, so P-Tree and Copernicus are unaffected.
- **"Logged in to Earthdata as: None", or "(token login - no username)".** You are authenticated, but not with the username and password you just gave — an `EARTHDATA_TOKEN` in the environment got there first, and `earthaccess` uses it in preference once it is already logged in. Downloads work, but this notebook has not verified your username and password. Print the variable — `import os; print(os.environ.get("EARTHDATA_TOKEN"))` — and if it shows anything, remove it from your Colab Secrets (the key icon in the left sidebar), then restart the runtime and re-run.
- **`401 Unauthorized`** from the Copernicus token cell, or `"Token not found"`. The email address or password is wrong — check them by logging in at <https://dataspace.copernicus.eu/> — or the account has two-factor authentication on, in which case the request needs the `totp` line in that cell as well.
- **`530 Login incorrect`** from JAXA. You used your website login rather than the FTP credentials, or your registration is not approved yet.
- **`KeyError`** in a check cell. Nothing signed you in to that provider — `ftp.ptree.jaxa.jp` or `identity.dataspace.copernicus.eu` — so add that provider's two secrets and run the setup cell at the top again.
- Anything else: [S01 Troubleshooting](https://github.com/earthobservatory/eyes-on-earth-tutorials/blob/main/S01_Troubleshooting/README.md).

## Next

[Tutorial 2 overview](https://github.com/earthobservatory/eyes-on-earth-tutorials/blob/main/2.0_Tutorial_2_Overview_and_Assignment/README.md)